# Hello World - 最初のAIエージェント

10行未満のコードで[Amazon Bedrock](https://aws.amazon.com/bedrock/)を使用して最初のAIエージェントを作成します。このノートブックでは、[Strands Agents SDK](https://github.com/strands-agents/sdk-python)を使用してエージェントの基本、システムプロンプト、実行ループを紹介します。

## 学習内容

- Amazon BedrockモデルでAIエージェントを作成する
- エージェントの動作のためにシステムプロンプトを設定する
- エージェントの会話と応答を実行する
- エージェントの実行ループを理解する

## 前提条件

- [Amazon Bedrockモデルアクセス](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)が有効なAWSアカウント
- 適切な権限で設定されたAWS CLI
- Python 3.9+環境

## セットアップ

まず、必要なライブラリをインポートします：

In [ ]:
import os
import boto3
from strands import Agent
from strands.models import BedrockModel

print("✅ Imports successful!")

## 最初のエージェントを作成する

数行のコードでシンプルなエージェントを作成しましょう：

In [ ]:
# Create a simple agent with default settings
agent = Agent()

print("✅ Agent created successfully!")
print("Understanding the Agent Interface")
# Check the current model being used by the agent
print(f"Current Agent model provider: {agent.model}")
print(f"Current Agent model id: {agent.model.config['model_id']}")
print("-" * 100)
print("\n")

## 最初の対話

エージェントとの最初の会話を始めましょう：

> 注：このHello Worldエージェントの例を実行するには、モデルプロバイダーの認証情報を設定し、モデルアクセスを有効にする必要があります。デフォルトのモデルプロバイダーはAmazon Bedrockで、デフォルトのモデルは認証情報のリージョンからClaude 4 Sonnet推論モデルです。例えば、リージョンをus-east-1に設定した場合、デフォルトのモデルIDは次のようになります：us.anthropic.claude-sonnet-4-20250514-v1:0。詳細は[Strands Agents Documentation](https://strandsagents.com/latest/documentation/docs/)を参照してください。

In [ ]:
# シンプルなHello World
response = agent("スペインの首都はどこですか？")

## エージェントの応答を理解する

エージェントは有用な情報を含む`AgentResult`オブジェクトを返します：

In [ ]:
# Inspect the AgentResult object
print(f"Message: {response.message}")
print("-" * 100 + "\n")

print(f"Metrics: {response.metrics}")
print("-" * 100 + "\n")

# print(f"State: {response.state}")
# print("-" * 100 + "\n")

# print(f"Stop Reason: {response.stop_reason}")
# print("-" * 100 + "\n")

## modelIDを使用してAWS Bedrockモデルを使用する

In [ ]:
print(f"✅ 現在のモデル: {agent.model.config['model_id']}")

# 特定のモデルでAWS Bedrockを設定する
session = boto3.Session(region_name='us-west-2') #セッションはAWS設定状態を保存し、サービスクライアントとリソースを作成できるようにします

bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-3-5-haiku-20241022-v1:0",
    boto_session=session,
    streaming=False
)

# エージェントをこのモデルを使用するように更新する
agent.model = bedrock_model

print(f"✅ モデルを更新しました: {agent.model.config['model_id']}")

## Anthropicモデルプロバイダーを使用する（オプション）

Amazon Bedrockの代わりにAnthropicのAPIを直接使用することもできます。

**前提条件**：Anthropic APIキーを環境変数として設定します：
```bash
export ANTHROPIC_API_KEY=your_api_key_here
```

In [ ]:
!pip install strands-agents[anthropic]

In [ ]:
from strands.models.anthropic import AnthropicModel

# 環境変数からAPIキーを取得する
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

if ANTHROPIC_API_KEY:
    # Anthropicモデルを設定する
    anthropic_model = AnthropicModel(
        client_args={"api_key": ANTHROPIC_API_KEY},
        model_id="claude-3-7-sonnet-20250219",
        max_tokens=1024,
        params={"temperature": 0.3}
    )
    
    # エージェントをAnthropicモデルを使用するように更新する
    agent.model = anthropic_model
    
    print(f"✅ Anthropicモデルに更新しました: {agent.model.config['model_id']}")
else:
    print("⚠️  ANTHROPIC_API_KEYが見つかりません。Anthropicモデルの設定をスキップします。")
    print("   次のコマンドで設定してください: export ANTHROPIC_API_KEY=your_api_key_here")

## エージェントの状態を確認する

エージェントのさまざまな側面を確認および変更できます：

In [ ]:
# 会話履歴を表示する
print(f"メッセージ: {agent.messages}")
print(f"メッセージ数: {len(agent.messages)}")

In [ ]:
# 現在のシステムプロンプトを確認する
print(f"システムプロンプト: {agent.system_prompt}")

In [ ]:
agent = Agent(model=bedrock_model, callback_handler=None)

# システムプロンプトを更新する
agent.system_prompt = "あなたは数学の問題についてのストーリーテラーです"
print(f"✅ システムプロンプトを更新しました: {agent.system_prompt}")

In [ ]:
# 更新されたシステムプロンプトをテストする
response = agent("フィボナッチ数列で2の次は何ですか？")
print(response)
print(f"\nメッセージ数: {len(agent.messages)}")

In [ ]:
# 会話履歴をクリアする
agent.messages.clear()
print(f"✅ メッセージをクリアしました。現在の数: {len(agent.messages)}")

## 専門化されたエージェントを作成する

異なる個性と専門知識を持つ複数のエージェントを作成できます：

In [ ]:
# 専門化されたAWSエキスパートエージェントを作成する
bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-3-5-haiku-20241022-v1:0",
    boto_session=session,
    streaming=False
)

aws_expert = Agent(
    model=bedrock_model,
    callback_handler=None,
    system_prompt="""あなたはAWSソリューションアーキテクトのエキスパートです。
    開発者がAWS上でクラウドソリューションを設計および実装するのを支援します。
    常に実用的で本番環境に対応したアドバイスを提供してください。"""
)

response = aws_expert("サーバーレスAPIを構築する最良の方法は何ですか？")
print(response)

## まとめ

このノートブックでは、以下を学習しました：

✅ 基本的なStrandsエージェントの作成方法

✅ Amazon Bedrockとの統合方法

✅ システムプロンプトでエージェントの動作をカスタマイズする方法

✅ エージェントとの対話方法


### 次のステップ

次のノートブックに進んで、カスタムツールの作成について学習しましょう！